In [2]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

In [3]:
from src.fraud.data import load_fraud_data, prepare_fraud_features, train_test_split_fraud

df_fraud = load_fraud_data()
df_fraud = prepare_fraud_features(df_fraud)

train_fraud, test_fraud = train_test_split_fraud(df_fraud)

print("Train :", len(train_fraud), "| Fraude :", train_fraud["fraud_label"].mean())
print("Test  :", len(test_fraud), "| Fraude :", test_fraud["fraud_label"].mean())

Train : 12336 | Fraude : 0.060797665369649805
Test  : 3084 | Fraude : 0.05609597924773022


In [4]:
from src.fraud.data import load_fraud_data, prepare_fraud_features, train_test_split_fraud

df_fraud = load_fraud_data()
df_fraud = prepare_fraud_features(df_fraud)

train_fraud, test_fraud = train_test_split_fraud(df_fraud)

print("Train :", len(train_fraud), "| Taux fraude :", train_fraud["fraud_label"].mean())
print("Test  :", len(test_fraud), "| Taux fraude :", test_fraud["fraud_label"].mean())

Train : 12336 | Taux fraude : 0.060797665369649805
Test  : 3084 | Taux fraude : 0.05609597924773022


In [5]:
from src.fraud.models import fit_isolation_forest, evaluate_isolation_forest

iso_model = fit_isolation_forest(train_fraud)
results_iso = evaluate_isolation_forest(iso_model, test_fraud)

print("AUC-ROC :", results_iso["auc_roc"])
print("PR-AUC  :", results_iso["pr_auc"])
print(f"\nPour référence, un modèle aléatoire aurait AUC-ROC ≈ 0.5 et PR-AUC ≈ {test_fraud['fraud_label'].mean():.4f} (prévalence)")

AUC-ROC : 0.5268197369753556
PR-AUC  : 0.0642208961843384

Pour référence, un modèle aléatoire aurait AUC-ROC ≈ 0.5 et PR-AUC ≈ 0.0561 (prévalence)


In [14]:
import pandas as pd
from src.fraud.models import fit_supervised_baseline, evaluate_supervised
from src.fraud.data import CATEGORICAL_COLS, NUMERIC_COLS

rf_model = fit_supervised_baseline(train_fraud)
results_rf = evaluate_supervised(rf_model, test_fraud)

print("=== Isolation Forest (non supervisé) ===")
print("AUC-ROC :", results_iso["auc_roc"], "| PR-AUC :", results_iso["pr_auc"])
print()
print("=== Random Forest (supervisé) ===")
print("AUC-ROC :", results_rf["auc_roc"], "| PR-AUC :", results_rf["pr_auc"])

# Importance des variables -- pour vérifier que AddressChange_Claim/Fault ressortent bien
importances = pd.Series(rf_model.feature_importances_, index=[c for c in train_fraud.columns if c.endswith(('_code','_norm'))])
print("\nTop 10 variables les plus importantes :")
print(importances.sort_values(ascending=False).head(10))

# Sauvegarder le modèle et les métadonnées pour l'API
import joblib

# Encodeurs : mêmes catégories, même ordre que ceux vus par astype("category") à l'entraînement
encoders = {
    col: df_fraud[col].astype("category").cat.categories.tolist()
    for col in CATEGORICAL_COLS
}
joblib.dump(encoders, "../models/fraud_encoders.pkl")

# Statistiques de normalisation (moyenne, écart-type) par variable numérique
normalization_stats = {
    col: (df_fraud[col].mean(), df_fraud[col].std())
    for col in NUMERIC_COLS
}
joblib.dump(normalization_stats, "../models/fraud_normalization_stats.pkl")

# Valeurs par défaut = mode pour les catégorielles, médiane pour les numériques
# (calculées sur train_fraud, jamais sur test)
default_values = {}
for col in CATEGORICAL_COLS:
    default_values[col] = train_fraud[col].mode()[0]
for col in NUMERIC_COLS:
    default_values[col] = train_fraud[col].median()
joblib.dump(default_values, "../models/fraud_default_values.pkl")

joblib.dump(rf_model, "../models/fraud_random_forest.pkl")

print("\n4 fichiers sauvegardés dans ../models/")
print("  - fraud_random_forest.pkl")
print("  - fraud_encoders.pkl")
print("  - fraud_normalization_stats.pkl")
print("  - fraud_default_values.pkl")

=== Isolation Forest (non supervisé) ===
AUC-ROC : 0.5268197369753556 | PR-AUC : 0.0642208961843384

=== Random Forest (supervisé) ===
AUC-ROC : 0.8147787046542614 | PR-AUC : 0.19127137858154528

Top 10 variables les plus importantes :
Fault_code                  0.241358
BasePolicy_code             0.172470
PolicyType_code             0.100040
VehicleCategory_code        0.076112
AddressChange_Claim_code    0.039970
Age_norm                    0.039020
Deductible_norm             0.033492
VehiclePrice_code           0.032665
Make_code                   0.029723
RepNumber_norm              0.024396
dtype: float64

4 fichiers sauvegardés dans ../models/
  - fraud_random_forest.pkl
  - fraud_encoders.pkl
  - fraud_normalization_stats.pkl
  - fraud_default_values.pkl


In [7]:
import sys
!{sys.executable} -m pip install torch_geometric

from src.fraud.graph import build_pyg_graph

feature_cols = [c for c in df_fraud.columns if c.endswith(("_code", "_norm")) and c not in ["RepNumber_code"]]

graph_data = build_pyg_graph(df_fraud, feature_cols, min_shared_attrs=2)

print("Nombre de nœuds :", graph_data.num_nodes)
print("Nombre d'arêtes (orientées, donc x2) :", graph_data.num_edges)
print("Degré moyen :", graph_data.num_edges / graph_data.num_nodes)
print()

# Vérification cruciale : les nœuds connectés partagent-ils plus souvent le label de fraude
# que des paires aléatoires ? (test d'homophilie -- condition nécessaire pour qu'un GNN apporte quelque chose)
edge_index = graph_data.edge_index.numpy()
y = graph_data.y.numpy()

if graph_data.num_edges > 0:
    same_label = (y[edge_index[0]] == y[edge_index[1]]).mean()
    print(f"Proportion de paires connectées avec le même label : {same_label:.4f}")
    print(f"Référence (accord attendu au hasard) : {(y.mean()**2 + (1-y.mean())**2):.4f}")

You should consider upgrading via the 'd:\Téléchargements\projet_actuariat\venv\Scripts\python.exe -m pip install --upgrade pip' command.


  Using cached multidict-6.7.1-cp310-cp310-win_amd64.whl (46 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
  Using cached frozenlist-1.8.0-cp310-cp310-win_amd64.whl (43 kB)


TypeError: build_edge_index() missing 1 required positional argument: 'similarity_cols'

In [ ]:
print("Distribution du nombre de dossiers par RepNumber :")
print(df_fraud.groupby("RepNumber").size().describe())

Distribution du nombre de dossiers par RepNumber :
count      16.000000
mean      963.750000
std        40.556956
min       892.000000
25%       941.750000
50%       961.500000
75%       986.250000
max      1069.000000
dtype: float64


In [ ]:

from src.fraud.graph import build_edge_index
similarity_cols = ["Fault", "AddressChange_Claim", "Days_Policy_Claim", "PolicyType", "BasePolicy"]

edge_index = build_edge_index(df_fraud, similarity_cols, min_shared_attrs=len(similarity_cols))

print("Nombre d'arêtes (orientées) :", edge_index.shape[1])
print("Degré moyen :", edge_index.shape[1] / len(df_fraud))

y = df_fraud["fraud_label"].values
if edge_index.shape[1] > 0:
    same_label = (y[edge_index[0].numpy()] == y[edge_index[1].numpy()]).mean()
    print(f"Proportion de paires connectées avec le même label : {same_label:.4f}")
    print(f"Référence hasard : {(y.mean()**2 + (1-y.mean())**2):.4f}")

Nombre d'arêtes (orientées) : 10544
Degré moyen : 0.6837872892347601
Proportion de paires connectées avec le même label : 0.8022
Référence hasard : 0.8875


In [ ]:

from src.fraud.graph import build_edge_index_rare_shared


cols_to_test = ["Fault", "AddressChange_Claim", "Days_Policy_Claim", "Days_Policy_Accident", "PastNumberOfClaims"]

edge_index = build_edge_index_rare_shared(df_fraud, cols_to_test, rarity_threshold=0.10)

print("Nombre d'arêtes (orientées) :", edge_index.shape[1])
print("Degré moyen :", edge_index.shape[1] / len(df_fraud))

y = df_fraud["fraud_label"].values
if edge_index.shape[1] > 0:
    same_label = (y[edge_index[0].numpy()] == y[edge_index[1].numpy()]).mean()
    print(f"Proportion de paires connectées avec le même label : {same_label:.4f}")
    print(f"Référence hasard : {(y.mean()**2 + (1-y.mean())**2):.4f}")

    # Détail : quelles valeurs rares génèrent le plus d'arêtes et avec quelle homophilie
    for col in cols_to_test:
        freq = df_fraud[col].value_counts(normalize=True)
        rare_values = freq[freq < 0.10].index
        print(f"\n{col} — valeurs rares (< 10%) : {list(rare_values)}")

Nombre d'arêtes (orientées) : 39750
Degré moyen : 2.577821011673152
Proportion de paires connectées avec le même label : 0.8575
Référence hasard : 0.8875

Fault — valeurs rares (< 10%) : []

AddressChange_Claim — valeurs rares (< 10%) : ['4 to 8 years', '2 to 3 years', '1 year', 'under 6 months']

Days_Policy_Claim — valeurs rares (< 10%) : ['15 to 30', '8 to 15', 'none']

Days_Policy_Accident — valeurs rares (< 10%) : ['none', '8 to 15', '15 to 30', '1 to 7']

PastNumberOfClaims — valeurs rares (< 10%) : []


In [11]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)

joblib.dump(rf_model, "../models/fraud_random_forest.pkl")

print("Model saved successfully!")

Model saved successfully!
